In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("PINPAD_DATA_ROOT", "/home/cse/pinpad/myenv/GNURadio_1/Correlation_Responses/OneSubcarrierLowDataRate"))
OUTPUT = Path(os.environ.get("PINPAD_OUTPUT_ROOT", "/home/cse/pinpad/myenv/GNURadio_1/PINPAD_Results"))
OUTPUT.mkdir(parents=True, exist_ok=True)
N_BUTTONS = 12
N_PER_BUTTON = 200
RESPONSE_LENGTH = 3624
SEED = 42

# ADAPT = {
#     "Source": [(c, "R1") for c in range(1, 6)],
#     "CC": [(c, "R1") for c in range(6, 11)],
#     "CR": [(c, "R2") for c in range(1, 6)],
#     "CCR": [(c, "R2") for c in range(11, 16)],
# }
TEST = {
    "CC": [(c, "R1") for c in range(16, 26)],
    "CR": [(c, "R3") for c in range(1, 6)],
    "CCR": [(c, "R2") for c in range(26, 36)],
}
assert sum(map(len, ADAPT.values())) == 20
assert sum(map(len, TEST.values())) == 25
assert not set(sum(ADAPT.values(), [])) & set(sum(TEST.values(), []))

MAP_FILE = ROOT / "collection_paths.csv"
expected = set(sum(ADAPT.values(), []) + sum(TEST.values(), []))
if MAP_FILE.is_file():
    manifest = pd.read_csv(MAP_FILE)
    assert {"card", "reader", "directory"}.issubset(manifest.columns)
    paths = {}
    for row in manifest.itertuples(index=False):
        key = (int(row.card), str(row.reader).upper())
        if key in paths:
            raise ValueError(f"Duplicate collection: {key}")
        p = Path(str(row.directory))
        paths[key] = p if p.is_absolute() else ROOT / p
else:
    paths = {}
    for card, reader in expected:
        candidates = [ROOT / reader / f"card{card}", ROOT / reader.lower() / f"card{card}",
                      ROOT / f"card{card}" / reader, ROOT / f"card{card}" / reader.lower()]
        hits = [p for p in candidates if p.is_dir()]
        if len(hits) == 1:
            paths[(card, reader)] = hits[0]
        elif len(hits) > 1:
            raise ValueError(f"Multiple directories for {(card, reader)}: {hits}")
missing = sorted(expected - set(paths))
if missing:
    pd.DataFrame([dict(card=c, reader=r, directory="") for c, r in sorted(expected)]).to_csv(
        OUTPUT / "collection_paths.csv", index=False)
    raise FileNotFoundError(f"Provide reader-specific directories in {MAP_FILE}; {len(missing)} collection paths missing. "
                            "A bare cardN folder cannot distinguish the same card on R1, R2 and R3.")

def load_group(keys):
    signals, labels, cards, readers, sample_ids = [], [], [], [], []
    for card, reader in keys:
        folder = paths[(card, reader)]
        for button in range(N_BUTTONS):
            filename = folder / f"button{button}.npy"
            if not filename.is_file():
                raise FileNotFoundError(filename)
            arr = np.load(filename, allow_pickle=False)
            if arr.shape != (N_PER_BUTTON, RESPONSE_LENGTH):
                raise ValueError(f"{filename}: expected {(N_PER_BUTTON, RESPONSE_LENGTH)}, got {arr.shape}")
            if not np.isfinite(arr).all():
                raise ValueError(f"Non-finite RF response: {filename}")
            signals.append(arr)
            labels.extend([button] * N_PER_BUTTON)
            cards.extend([card] * N_PER_BUTTON)
            readers.extend([reader] * N_PER_BUTTON)
            sample_ids.extend(range(N_PER_BUTTON))
    return (np.concatenate(signals), np.asarray(labels, dtype=int), np.asarray(cards, dtype=int),
            np.asarray(readers), np.asarray(sample_ids, dtype=int))

samples = {domain: load_group(keys) for domain, keys in TEST.items()}
assert sum(len(t[1]) for t in samples.values()) == 60000
for domain, (X, y, cards, readers, sample_ids) in samples.items():
    assert len(y) == {"CC": 24000, "CR": 12000, "CCR": 24000}[domain]
    assert set(cards) == {"CC": set(range(16, 26)), "CR": set(range(1, 6)),
                          "CCR": set(range(26, 36))}[domain]
    assert set(readers) == {"CC": {"R1"}, "CR": {"R3"}, "CCR": {"R2"}}[domain]
    assert set(y) == set(range(N_BUTTONS))


In [ ]:
import joblib
import tensorflow as tf
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split
from mapie.classification import SplitConformalClassifier

ALPHA = {"DL": 0.07, "RF": 0.05}
TAU = 1.0
MAX_CAPTURES = 3
SCORE = os.environ.get("PINPAD_CP_SCORE", "aps").lower()
if SCORE not in ("aps", "lac"):
    raise ValueError("PINPAD_CP_SCORE must be aps or lac")

DL_PATH = Path(os.environ.get("PINPAD_DL_MODEL", str(OUTPUT / "DL_CORAL.keras")))
RF_PATH = Path(os.environ.get("PINPAD_RF_MODEL", str(OUTPUT / "RF_models.joblib")))
if not DL_PATH.is_file() or not RF_PATH.is_file():
    raise FileNotFoundError(f"Missing trained models: {DL_PATH}; {RF_PATH}")
dl_model = tf.keras.models.load_model(DL_PATH, compile=False)
rf_bundle = joblib.load(RF_PATH)
pca, rf_model = rf_bundle["pca"], rf_bundle["rf"]
if not np.array_equal(rf_model.classes_, np.arange(N_BUTTONS)):
    raise ValueError("RF class order must be B0 to B11")

def input_for_dl(X):

    mode = os.environ.get("PINPAD_DL_INPUT_FORMAT", "").lower()
    if mode == "magnitude":
        v = np.abs(X).astype(np.float32)[..., None]
    elif mode == "real_imag":
        v = np.stack([X.real, X.imag], axis=-1).astype(np.float32)
    elif mode == "real":
        v = X.real.astype(np.float32)[..., None]
    else:
        raise ValueError("Set PINPAD_DL_INPUT_FORMAT to magnitude, real_imag or real based on original trained-model preprocessing")
    if tuple(dl_model.input_shape[1:]) != tuple(v.shape[1:]):
        raise ValueError(f"DL model expects {dl_model.input_shape}; supplied {v.shape}")
    return v

def input_for_rf(X):
    mode = os.environ.get("PINPAD_RF_INPUT_FORMAT", "").lower()
    if mode == "magnitude":
        v = np.abs(X).astype(np.float32)
    elif mode == "real_imag":
        v = np.concatenate([X.real, X.imag], axis=1).astype(np.float32)
    elif mode == "real":
        v = X.real.astype(np.float32)
    else:
        raise ValueError("Set PINPAD_RF_INPUT_FORMAT to magnitude, real_imag or real based on original fitted PCA preprocessing")
    if v.shape[1] != pca.n_features_in_:
        raise ValueError(f"PCA requires {pca.n_features_in_} input features, received {v.shape[1]}")
    return v

class PretrainedDL(ClassifierMixin, BaseEstimator):
    def __init__(self, model):
        self.model = model
        self.classes_ = np.arange(N_BUTTONS)
    def __sklearn_is_fitted__(self):
        return True
    def predict_proba(self, X):
        scores = np.asarray(self.model.predict(input_for_dl(X), batch_size=256, verbose=0))
        if scores.shape != (len(X), N_BUTTONS):
            raise ValueError(f"Invalid DL output: {scores.shape}")
        if np.any(scores < 0) or not np.isfinite(scores).all() or not np.allclose(scores.sum(1), 1, atol=1e-3):
            raise ValueError("DL outputs must be 12 probabilities; check whether the model returns logits")
        return scores
    def predict(self, X):
        return self.predict_proba(X).argmax(1)

class PretrainedRF(ClassifierMixin, BaseEstimator):
    def __init__(self, pca, model):
        self.pca, self.model = pca, model
        self.classes_ = np.asarray(model.classes_)
    def __sklearn_is_fitted__(self):
        return True
    def predict_proba(self, X):
        return self.model.predict_proba(self.pca.transform(input_for_rf(X)))
    def predict(self, X):
        return self.predict_proba(X).argmax(1)

models = {"DL": PretrainedDL(dl_model), "RF": PretrainedRF(pca, rf_model)}


In [ ]:
def split_data(y, cards, sample_ids):
    # Keep captures from each 50-response placement block in one partition.
    keys = np.asarray([f"{c}:{b}:{s // 50}" for c, b, s in zip(cards, y, sample_ids)])
    unique = np.unique(keys)

    from sklearn.model_selection import GroupShuffleSplit
    outer = GroupShuffleSplit(n_splits=1, test_size=0.8, random_state=SEED)
    first, test = next(outer.split(y, y, groups=keys))
    inner = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
    a, b = next(inner.split(y[first], y[first], groups=keys[first]))
    calibration, validation = first[a], first[b]
    if len(set(keys[calibration]) & set(keys[test])) or len(set(keys[validation]) & set(keys[test])):
        raise ValueError("Coil-placement blocks leaked between splits")
    if not (0.08 <= len(calibration)/len(y) <= 0.12 and 0.08 <= len(validation)/len(y) <= 0.12):
        raise ValueError("Unexpected partition sizes")
    return calibration, validation, test

def fit_mapie(model, X_cal, y_cal, alpha):
    cp = SplitConformalClassifier(estimator=model, confidence_level=1-alpha,
                                  conformity_score=SCORE, prefit=True, random_state=SEED)
    cp.conformalize(X_cal, y_cal)
    return cp

def prediction_sets(cp, X):
    _, sets = cp.predict_set(X)
    sets = np.asarray(sets, dtype=bool)
    if sets.ndim == 3 and sets.shape[-1] == 1:
        sets = sets[..., 0]
    if sets.shape != (len(X), N_BUTTONS):
        raise ValueError(f"Unexpected MAPIE set shape: {sets.shape}")
    return sets

def assess(sets, y):
    return float(sets[np.arange(len(y)), y].mean()), float(sets.sum(1).mean())

partitions = {domain: split_data(y, cards, sample_ids)
              for domain, (_, y, cards, _, sample_ids) in samples.items()}


In [ ]:
calibrators = {}

def evaluate_model(name):
    rows = []
    for domain, (X, y, cards, readers, sample_ids) in samples.items():
        calibration, validation, test = partitions[domain]
        cp = fit_mapie(models[name], X[calibration], y[calibration], ALPHA[name])
        calibrators[name, domain] = cp
        vsets = prediction_sets(cp, X[validation])
        tsets = prediction_sets(cp, X[test])
        vcover, vsize = assess(vsets, y[validation])
        coverage, size = assess(tsets, y[test])
        probs = models[name].predict_proba(X[test])
        accuracy = float((probs.argmax(1) == y[test]).mean())
        np.savez_compressed(OUTPUT / f"{name}_{domain}_MAPIE.npz", prediction_sets=tsets,
                            probabilities=probs, labels=y[test], card=cards[test],
                            reader=readers[test], sample_id=sample_ids[test],
                            heldout_index=test, calibration_index=calibration, validation_index=validation)
        rows.append(dict(model=name, domain=domain, alpha=ALPHA[name], conformity_score=SCORE,
                         calibration=len(calibration), validation=len(validation), testing=len(test),
                         validation_coverage=vcover, validation_set_size=vsize,
                         coverage=coverage, prediction_set_size=size, single_accuracy=accuracy))
    report = pd.DataFrame(rows)
    report.to_csv(OUTPUT / f"{name}_MAPIE_domain_metrics.csv", index=False)
    print(report.to_string(index=False))
    return report


In [ ]:
risk_levels = np.array([0.05, 0.07, 0.10, 0.15, 0.20, 0.25, 0.30])
thresholds = np.arange(0.50, 1.001, 0.10)
capture_budgets = (1, 2, 3)

def validation_sweep(name):
    records = []
    for domain, (X, y, cards, readers, sample_ids) in samples.items():
        calibration, validation, test = partitions[domain]
        for alpha in risk_levels:
            cp = fit_mapie(models[name], X[calibration], y[calibration], float(alpha))
            sets = prediction_sets(cp, X[validation])
            coverage, size = assess(sets, y[validation])
            records.append(dict(model=name, domain=domain, alpha=alpha,
                                coverage=coverage, prediction_set_size=size,
                                validation_samples=len(validation)))
    result = pd.DataFrame(records)
    result.to_csv(OUTPUT / f"{name}_validation_risk_sweep.csv", index=False)
    return result


def validation_adaptive_sweep(name, validation_episodes):
    needed = {"domain", "episode_id", "capture_index", "row_index", "card", "reader", "button"}
    if not needed.issubset(validation_episodes):
        raise ValueError(f"Missing validation columns: {needed - set(validation_episodes)}")
    rows = []
    for domain, (X, y, cards, readers, sample_ids) in samples.items():
        cal, val, test = partitions[domain]
        lookup = {int(row): idx for idx, row in enumerate(val)}
        e = validation_episodes[validation_episodes.domain == domain]
        if "model" in e.columns:
            e = e[e.model == name]
        for alpha in risk_levels:
            cp = fit_mapie(models[name], X[cal], y[cal], float(alpha))
            psets = prediction_sets(cp, X[val])
            for tau in thresholds:
                for budget in capture_budgets:
                    scores = []
                    for episode_id, captures in e.groupby("episode_id", sort=False):
                        captures = captures.sort_values("capture_index")
                        rr = captures.row_index.astype(int).to_list()
                        if not all(i in lookup for i in rr):
                            raise ValueError(f"Non-validation episode: {episode_id}")
                        if not 1 <= len(rr) <= MAX_CAPTURES:
                            raise ValueError(f"Invalid capture count: {episode_id}")
                        if not np.array_equal(captures.capture_index.to_numpy(), np.arange(1, len(rr)+1)):
                            raise ValueError(f"Unordered episode: {episode_id}")
                        positions = [lookup[i] for i in rr[:budget]]
                        original_rows = np.asarray(rr[:budget])
                        if len(set(original_rows)) != len(original_rows):
                            raise ValueError(f"Duplicate response in {episode_id}")
                        if len(set(y[val][positions])) != 1 or len(set(cards[val][positions])) != 1:
                            raise ValueError(f"Mixed buttons/cards in {episode_id}")
                        ev = np.zeros(N_BUTTONS)
                        used, pred = budget, -1
                        for k, mask in enumerate(psets[positions], 1):
                            active = np.flatnonzero(mask)
                            if len(active):
                                ev[active] += 1 / len(active)
                            if ev.max() >= tau:
                                used, pred = k, int(np.argmax(ev))
                                break
                        if pred < 0:
                            if len(positions) < budget:
                                raise ValueError(f"Episode {episode_id} has fewer captures than required for budget {budget}")
                            pred = int(np.argmax(ev))
                        scores.append((pred == y[val][positions[0]], used-1))
                    if scores:
                        rows.append(dict(model=name, domain=domain, alpha=alpha, tau=round(float(tau), 2),
                                         max_captures=budget, accuracy=np.mean([a for a,b in scores]),
                                         mean_reinputs=np.mean([b for a,b in scores]), episodes=len(scores)))
    if not rows:
        raise ValueError("No independent validation capture sequences available")
    result = pd.DataFrame(rows)
    result.to_csv(OUTPUT / f"{name}_validation_adaptive_sweep.csv", index=False)
    return result

validation_sequence_path = Path(os.environ.get("PINPAD_VALIDATION_SEQUENCES", str(OUTPUT / "validation_capture_sequences.csv")))
if validation_sequence_path.is_file():
    validation_episodes = pd.read_csv(validation_sequence_path)
else:
    validation_episodes = None


In [ ]:
# dl_validation_sweep = validation_sweep("DL")
# dl_validation_sweep

In [ ]:
# rf_validation_sweep = validation_sweep("RF")
# rf_validation_sweep

In [ ]:
# dl_validation_adaptive = validation_adaptive_sweep("DL", validation_episodes) if validation_episodes is not None else None

In [ ]:
# rf_validation_adaptive = validation_adaptive_sweep("RF", validation_episodes) if validation_episodes is not None else None

In [ ]:
# dl_results = evaluate_model("DL")

In [ ]:
rf_results = evaluate_model("RF")

In [ ]:
def adaptive_decision(capture_sets):
    evidence = np.zeros(N_BUTTONS, dtype=float)
    for capture_count, mask in enumerate(capture_sets[:MAX_CAPTURES], start=1):
        buttons = np.flatnonzero(mask)
        if len(buttons):
            evidence[buttons] += 1.0 / len(buttons)
        maximum = evidence.max()
        if maximum >= TAU:
            return int(np.argmax(evidence)), capture_count, True
    if len(capture_sets) != MAX_CAPTURES:
        raise ValueError("A full three-capture sequence is required when no earlier acceptance occurs")
    return int(np.argmax(evidence)), MAX_CAPTURES, False

def evaluate_adaptive(name, episodes):
    results = []
    required = {"domain", "episode_id", "capture_index", "row_index", "card", "reader", "button"}
    if not required <= set(episodes.columns):
        raise ValueError(f"Missing capture fields: {required - set(episodes.columns)}")
    for domain in TEST:
        rows = episodes[episodes.domain == domain]
        if "model" in episodes.columns:
            rows = rows[rows.model == name]
        archive = np.load(OUTPUT / f"{name}_{domain}_MAPIE.npz", allow_pickle=False)
        indices = archive["heldout_index"]
        by_original = {int(idx): position for position, idx in enumerate(indices)}
        masks, labels, cards, readers = (archive[k] for k in ("prediction_sets", "labels", "card", "reader"))
        for episode_id, group in rows.groupby("episode_id", sort=False):
            group = group.sort_values("capture_index")
            if not np.array_equal(group.capture_index.to_numpy(), np.arange(1, len(group)+1)):
                raise ValueError(f"Nonconsecutive captures: {domain}, {episode_id}")
            if not 1 <= len(group) <= MAX_CAPTURES:
                raise ValueError(f"Invalid capture count: {domain}, {episode_id}")
            original_indices = group.row_index.astype(int).to_list()
            if not all(idx in by_original for idx in original_indices):
                raise ValueError(f"Episode contains non-held-out capture: {domain}, {episode_id}")
            positions = [by_original[idx] for idx in original_indices]
            if len(set(original_indices)) != len(original_indices):
                raise ValueError(f"Repeated RF response in episode {episode_id}")
            if not (np.all(labels[positions] == labels[positions[0]]) and
                    np.all(cards[positions] == cards[positions[0]]) and
                    np.all(readers[positions] == readers[positions[0]])):
                raise ValueError(f"Mixed button/card/reader in episode {episode_id}")
            if not (np.all(group.button.to_numpy() == labels[positions]) and
                    np.all(group.card.to_numpy() == cards[positions]) and
                    np.all(group.reader.astype(str).str.upper().to_numpy() == readers[positions])):
                raise ValueError(f"Capture metadata mismatch: {domain}, {episode_id}")
            prediction, count, accepted = adaptive_decision(masks[positions])
            results.append(dict(model=name, domain=domain, episode_id=episode_id,
                                correct=(prediction == int(labels[positions[0]])),
                                captures=count, reinputs=count-1, accepted=accepted))
    if not results:
        raise ValueError("No measured capture episodes found; cannot compute adaptive metrics")
    frame = pd.DataFrame(results)
    frame.to_csv(OUTPUT / f"{name}_adaptive_episode_results.csv", index=False)
    print(frame.groupby("domain").agg(accuracy=("correct", "mean"),
          mean_reinputs=("reinputs", "mean"), count=("episode_id", "size")))
    return frame

sequence_path = Path(os.environ.get("PINPAD_CAPTURE_SEQUENCES", str(OUTPUT / "capture_sequences.csv")))
if sequence_path.is_file():
    episodes = pd.read_csv(sequence_path)
else:
    episodes = None
